In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml import command, load_component
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import Input, Output
from azureml.core import Workspace, Dataset, Datastore

subscription_id = '813ac1cb-7b8e-45ab-9b3c-8510079c761a'
resource_group = 'MIBLab_Official'
workspace_name = 'miblab-official-ml'

workspace = Workspace(subscription_id, resource_group, workspace_name)
datastore = Datastore.get(workspace, "eegfoundationmodeldata")

# Get a handle to the workspace
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)

cluster_name = "miblab-E16ads-v5-cpu-dedicated"
conversion_component = load_component(source="./convertTUHtoBIDS.yml")

In [ ]:
input_dir = Input(
	type="uri_folder",
	mode='ro_mount',
	path="azureml://subscriptions/813ac1cb-7b8e-45ab-9b3c-8510079c761a/resourcegroups/MIBLab_Official/workspaces/miblab-official-ml/datastores/eegfoundationmodeldata/paths/edf/"
)

output_dir = Output(
	type="uri_folder",
	path='azureml://subscriptions/813ac1cb-7b8e-45ab-9b3c-8510079c761a/resourcegroups/MIBLab_Official/workspaces/miblab-official-ml/datastores/eegfoundationmodeldata/paths/eegdata/TUH_BIDS_Converted',
	mode='rw_mount'
)


In [ ]:
# Define a pipeline with component
@pipeline(default_compute=cluster_name)
def conversion(input_dir):
	conversion= conversion_component(input_dir=input_dir)
	return {
        "output_dir": conversion.outputs.output_dir
    }

In [ ]:
conversion_job = conversion(input_dir=input_dir)
conversion_job.outputs.output_dir = output_dir

In [ ]:
# submit job to workspace
conversion_job = ml_client.jobs.create_or_update(
	conversion_job,
    experiment_name="EEGFoundationModel",
)

conversion_job